In [1]:
import pandas as pd
import re
import unidecode
import os
import glob
from datetime import datetime

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
# pegando os arquivos mais recentes
def arquivo_mais_recente(padrao):
    arquivos = glob.glob(padrao)
    if not arquivos:
        raise FileNotFoundError(f"Nenhum arquivo encontrado para o padrão: {padrao}")
    return max(arquivos, key=os.path.getctime)


In [3]:
amazon_csv = arquivo_mais_recente("data/resultados_amazon/mais_vendidos_amazon_SAMPLE_*.csv")
ml_csv = arquivo_mais_recente("data/resultados_ml/mais_vendidos_ml_SAMPLE_*.csv")

print("📄 Arquivo Amazon:", amazon_csv)
print("📄 Arquivo Mercado Livre:", ml_csv)

amazon = pd.read_csv(amazon_csv, sep=';', encoding='utf-8-sig')
mercadolivre = pd.read_csv(ml_csv, sep=';', encoding='utf-8-sig')


📄 Arquivo Amazon: data/resultados_amazon\mais_vendidos_amazon_SAMPLE_2026-03-d_18-49.csv
📄 Arquivo Mercado Livre: data/resultados_ml\mais_vendidos_ml_SAMPLE_2026-03-d_13-00.csv


In [4]:
col_nome_amz = "Nome"
col_nome_ml = "Nome"

col_cat_amz = "Subcategoria"
col_cat_ml = "Subcategoria"

col_nota = "Nota Geral"
col_qtd = "Qtd. Avaliações"



In [5]:
# limpando os textos
def limpar_texto(texto):
    texto = str(texto).lower()
    texto = unidecode.unidecode(texto)
    texto = re.sub(r'[^a-z0-9\s]', '', texto)
    return texto.strip()


In [6]:
def cortar_nome(texto, limite=60):
    return texto[:limite]


In [7]:
amazon["nome_match"] = (
    amazon[col_nome_amz]
    .apply(limpar_texto)
    .apply(lambda x: cortar_nome(x, 60))
)

mercadolivre["nome_match"] = (
    mercadolivre[col_nome_ml]
    .apply(limpar_texto)
    .apply(lambda x: cortar_nome(x, 60))
)

# combinando categoria e nome para melhorar a similaridade

In [8]:
### MAPA DE CATEGORIAS RELACIONADAS
MAPA_CATEGORIAS = {
    "Alimentos e Bebidas": ["Alimentos e Bebidas", "Esportes e Fitness", "Saúde"],
    "Automotivo": ["Acessórios para Veículos"],
    "Bebês": ["Bebês"],
    "Beleza": ["Beleza e Cuidado Pessoal"],
    "Beleza Premium": ["Beleza e Cuidado Pessoal"],
    "Brinquedos e Jogos": ["Brinquedos e Hobbies"],
    "Casa": ["Casa, Móveis e Decoração", "Agro"],
    "CD e Vinil": ["Música, Filmes e Seriados"],
    "Computadores e Informática": [
        "Informática", "Eletrônicos, Áudio e Vídeo", "Câmeras e Acessórios"
    ],
    "Cozinha": ["Eletrodomésticos", "Casa, Móveis e Decoração"],
    "Dispositivos Amazon e Acessórios": [
        "Informática", "Eletrônicos, Áudio e Vídeo"
    ],
    "DVD e Blu-ray": ["Música, Filmes e Seriados"],
    "Eletrodomésticos": ["Eletrodomésticos", "Casa, Móveis e Decoração"],
    "Eletrônicos": [
        "Eletrônicos, Áudio e Vídeo",
        "Celulares e Telefones",
        "Câmeras e Acessórios"
    ],
    "Esporte": ["Esportes e Fitness", "Saúde"],
    "Ferramentas e Mat. de Construção": ["Ferramentas", "Construção", "Agro"],
    "Games e Consoles": ["Games"],
    "Instrumentos Musicais": ["Instrumentos Musicais"],
    "Jardim e Piscina": ["Casa, Móveis e Decoração", "Agro"],
    "Livros": ["Livros, Revistas e Comics"],
    "Moda": ["Calçados, Roupas e Bolsas", "Joias e Relógios"],
    "Móveis": ["Casa, Móveis e Decoração"],
    "Papelaria e Escritório": ["Arte, Papelaria e Armarinho"],
    "Pet Shop": ["Pet Shop"],
    "Saúde e Bem-Estar": ["Saúde", "Beleza e Cuidado Pessoal", "Esportes e Fitness"]
}


In [9]:
def media_ponderada(nota_amz, qtd_amz, nota_ml, qtd_ml):
    total = qtd_amz + qtd_ml

    if total == 0:
        return None

    return round(
        (nota_amz * qtd_amz + nota_ml * qtd_ml) / total,
        2
    )


In [10]:
def converter_nota(valor):
    if pd.isna(valor):
        return 0.0
    return float(str(valor).replace(',', '.'))


In [11]:
def converter_qtd(valor):
    if pd.isna(valor):
        return 0
    valor = re.sub(r'\D', '', str(valor))
    return int(valor) if valor else 0


In [12]:
def montar_lookup_preco_amazon(df_amz_all):
    df = df_amz_all.copy()
    df["ASIN"] = df["ASIN"].astype(str).str.strip().str.upper()
    df["Preço"] = df["Preço"].astype(str).str.strip()

    # se tiver coluna com data do scraping, melhor ordenar por ela.
    # como normalmente não tem, a gente resolve pegando "última ocorrência" no concat.
    df = df[df["ASIN"].str.lower() != "nan"]
    df = df[df["Preço"].str.lower() != "nan"]

    # último preço por ASIN
    return df.drop_duplicates(subset=["ASIN"], keep="last").set_index("ASIN")["Preço"].to_dict()

def montar_lookup_preco_ml(df_ml_all):
    df = df_ml_all.copy()
    df["ASIN"] = df["ASIN"].astype(str).str.strip().str.upper()
    df["Preço à vista"] = df["Preço à vista"].astype(str).str.strip()

    df = df[df["ASIN"].str.lower() != "nan"]
    df = df[df["Preço à vista"].str.lower() != "nan"]

    return df.drop_duplicates(subset=["ASIN"], keep="last").set_index("ASIN")["Preço à vista"].to_dict()

In [13]:
from utils.loaders import carregar_todos_amazon, carregar_todos_ml

df_amz_all = carregar_todos_amazon()
df_ml_all = carregar_todos_ml()

lookup_preco_amz = montar_lookup_preco_amazon(df_amz_all)
lookup_preco_ml  = montar_lookup_preco_ml(df_ml_all)

In [14]:
lookup_preco_ml_link = (
    df_ml_all.assign(Link=df_ml_all["Link"].astype(str).str.strip())
             .dropna(subset=["Link"])
             .drop_duplicates(subset=["Link"], keep="last")
             .set_index("Link")["Preço à vista"]
             .to_dict()
)

In [15]:
vectorizer = TfidfVectorizer()


In [17]:
matches = []

for categoria_amz, categorias_ml in MAPA_CATEGORIAS.items():

    amazon_cat = amazon[amazon[col_cat_amz] == categoria_amz]
    ml_cat = mercadolivre[mercadolivre[col_cat_ml].isin(categorias_ml)]

    if amazon_cat.empty or ml_cat.empty:
        continue

    tfidf_amz = vectorizer.fit_transform(amazon_cat["nome_match"])
    tfidf_ml = vectorizer.transform(ml_cat["nome_match"])


    similarity_matrix = cosine_similarity(tfidf_amz, tfidf_ml)

    for i, (_, prod_amz) in enumerate(amazon_cat.iterrows()):
        idx_best = similarity_matrix[i].argmax()
        prod_ml = ml_cat.iloc[idx_best]

        # Amazon
        nota_amz = converter_nota(prod_amz[col_nota])
        qtd_amz = converter_qtd(prod_amz[col_qtd])

        # Mercado Livre
        nota_ml = converter_nota(prod_ml[col_nota])
        qtd_ml = converter_qtd(prod_ml[col_qtd])


        nota_final = media_ponderada(
            nota_amz, qtd_amz,
            nota_ml, qtd_ml
        )

        asin_amz = str(prod_amz["ASIN"]).strip().upper()
        asin_ml  = str(prod_ml.get("ASIN", "")).strip().upper()

        preco_amz = lookup_preco_amz.get(asin_amz, None)
        preco_ml  = lookup_preco_ml.get(asin_ml, None)

        matches.append({
            "Categoria Amazon": categoria_amz,
            "ASIN Amazon": prod_amz["ASIN"],
            "Produto Amazon": prod_amz[col_nome_amz],
            "Preço Prod Amazon": preco_amz,
            "Nota Amazon": nota_amz,
            "Avaliações Amazon": qtd_amz,
            "Link Amazon": prod_amz["Link"],

            "ASIN Mercado Livre": prod_ml.get("ASIN", None),
            "Produto Mercado Livre": prod_ml[col_nome_ml],
            "Preço Prod ML": preco_ml,
            "Nota Mercado Livre": nota_ml,
            "Avaliações ML": qtd_ml,
            "Link Mercado Livre": prod_ml["Link"],

            "Similaridade": round(similarity_matrix[i, idx_best], 3),
            "Nota Geral Ponderada": nota_final
        })

In [18]:
resultados = pd.DataFrame(matches)


In [19]:
limiar = 0.70
resultados_filtrados = resultados[resultados["Similaridade"] >= limiar]

print("🧾 Resultados filtrados:")
print(resultados_filtrados)


🧾 Resultados filtrados:
                Categoria Amazon ASIN Amazon  \
0            Alimentos e Bebidas  B097BYXGXN   
16           Alimentos e Bebidas  B00P9VQCXY   
38           Alimentos e Bebidas  B0BKH9MZ5Q   
108                        Bebês  B07GQ136TS   
112                        Bebês  B08K4L452V   
113                        Bebês  B093G7B8CN   
179                       Beleza  B0DS692QJR   
222               Beleza Premium  B07XP8K162   
242           Brinquedos e Jogos  B0DFFTP71B   
272           Brinquedos e Jogos  B0F2LML91K   
417   Computadores e Informática  B0F1G2TL8P   
434                      Cozinha  B0F256HPZJ   
440                      Cozinha  B07QK91PTZ   
453                      Cozinha  B09BK73232   
472                      Cozinha  B0G6XGZQW2   
635                  Eletrônicos  B0DJFTJ6LX   
643                  Eletrônicos  B0DYVPCX34   
645                  Eletrônicos  B0FCYPQ91Q   
653                  Eletrônicos  B0FR5GNJDB   
656             

In [ ]:
# saida_csv = f"data/comparacoes/comparacao_categorias_{datetime.now().strftime('%Y-%m-%d_%H-%M')}.csv"

# resultados_filtrados.to_csv(
#     saida_csv,
#     index=False,
#     sep=';',
#     encoding='utf-8-sig'
# )

# print(f"✅ Resultado salvo em: {saida_csv}")


In [20]:
# =========================
# PARTE AJUSTADA: Merge com UPDATE dos preços, avaliações e notas
#    agora trata cada lado (Amazon / Mercado Livre) independentemente
#    valores são colocados em PREÇO PROD Amazon/ML; colunas "Preço Amazon" e
#    "Preço Mercado Livre" serão descartadas se existirem.
# =========================

# 1) Acha o último MASTER (se existir)
padrao_master = "data/comparacoes/comparacao_categorias_MASTER_*.csv"
masters = glob.glob(padrao_master)

df_master = None
if masters:
    ultimo_master = max(masters, key=os.path.getctime)
    print("📌 Último MASTER encontrado:", ultimo_master)
    df_master = pd.read_csv(ultimo_master, sep=';', encoding='utf-8-sig')
else:
    print("ℹ️ Nenhum MASTER encontrado ainda. Vou criar um novo a partir do df atual.")

# 2) Se houver MASTER, fazer merge com UPDATE (não duplicação)
if df_master is not None and not df_master.empty:
    df_consolidado = df_master.copy()

    # colunas que atualizamos em cada plataforma
    amazon_cols = ["Preço Prod Amazon", "Nota Amazon", "Avaliações Amazon"]
    ml_cols = ["Preço Prod ML", "Nota Mercado Livre", "Avaliações ML"]

    # elimina colunas antigas de preço caso existam
    for col_old in ["Preço Amazon", "Preço Mercado Livre"]:
        if col_old in df_consolidado.columns:
            df_consolidado.drop(columns=[col_old], inplace=True)

    for _, row_novo in resultados_filtrados.iterrows():
        prod_amz = row_novo.get("Produto Amazon", "")
        prod_ml = row_novo.get("Produto Mercado Livre", "")

        updated = False

        # 2a) atualiza linhas com mesmo nome de produto Amazon
        if prod_amz != "" and "Produto Amazon" in df_consolidado.columns:
            mask_amz = df_consolidado["Produto Amazon"] == prod_amz
            if mask_amz.any():
                for col in amazon_cols:
                    if col in row_novo.index and col in df_consolidado.columns:
                        df_consolidado.loc[mask_amz, col] = row_novo[col]
                updated = True

        # 2b) atualiza linhas com mesmo nome de produto Mercado Livre
        if prod_ml != "" and "Produto Mercado Livre" in df_consolidado.columns:
            mask_ml = df_consolidado["Produto Mercado Livre"] == prod_ml
            if mask_ml.any():
                for col in ml_cols:
                    if col in row_novo.index and col in df_consolidado.columns:
                        df_consolidado.loc[mask_ml, col] = row_novo[col]
                updated = True

        # 2c) se não atualizou nada, adiciona como novo registro
        if not updated:
            df_consolidado = pd.concat(
                [df_consolidado, pd.DataFrame([row_novo])],
                ignore_index=True
            )
            print(f"  ➕ Adicionado: {prod_amz} ↔ {prod_ml}")
        else:
            print(f"  ✏️ Informações atualizadas para: {prod_amz} / {prod_ml}")
else:
    # sem master anterior: somente resultados de hoje
    df_consolidado = resultados_filtrados.copy()
    print("  (Nenhum MASTER prévio, usando resultados de hoje)")

# 3) Salva novo MASTER
saida_master = f"data/comparacoes/comparacao_categorias_MASTER_{datetime.now().strftime('%Y-%m-%d_%H-%M')}.csv"
df_consolidado.to_csv(saida_master, index=False, sep=';', encoding='utf-8-sig')

print(f"\n🧩 Consolidado (atualizado). Total linhas: {len(df_consolidado)}")
print(f"✅ MASTER salvo em: {saida_master}")

📌 Último MASTER encontrado: data/comparacoes\comparacao_categorias_MASTER_2026-03-27_14-52.csv
  ✏️ Informações atualizadas para: Heinz Ketchup Tradicional 1,033KG / Ketchup Tradicional 1,033kg Heinz
  ✏️ Informações atualizadas para: Azeite De Oliva Extra Virgem Andorinha - 500ml / Azeite De Oliva Extra Virgem Andorinha 500ml
  ➕ Adicionado: Dr. Peanut Pasta De Amendoim Avelã 250g - Com Whey Protein ↔ Pasta De Amendoim Com Whey Protein Chocolate Branco 600g Dr. Peanut
  ➕ Adicionado: Fralda Pampers Premium Care Tamanho P 40 Unidades ↔ Fralda Pampers Premium Care Tamanho M 80 Unidades
  ➕ Adicionado: Fralda Pampers Premium Care Pants Tamanho XXG, Fácil de Vestir, 90 Unidades ↔ Fralda Pampers Premium Care Pants Tamanho Xxg 90 Unidades
  ✏️ Informações atualizadas para: Fralda Pom Pom Protek Proteção de Mãe Jumbo P 30 Unidades / Fralda Pom Pom Protek Proteção De Mãe Jumbo G Com 24un Sem Gênero
  ✏️ Informações atualizadas para: GLOSS FRAN BY FRANCINY EHLKE LIPHONEY / Gloss Fran By Franci

In [ ]:
# import pandas as pd
# import re
# import unidecode
# import numpy as np

# # =========================================================
# # 1) FUNÇÕES AUXILIARES
# # =========================================================

# def limpar_texto_attr(texto):
#     texto = str(texto).lower()
#     texto = unidecode.unidecode(texto)
#     texto = re.sub(r'[^a-z0-9\s]', ' ', texto)
#     texto = re.sub(r'\s+', ' ', texto).strip()
#     return texto


# STOPWORDS_TITULO = {
#     "com", "para", "de", "da", "do", "e", "em", "a", "o",
#     "original", "novo", "nova", "kit", "premium", "plus",
#     "unidade", "unidades", "refil", "pack", "preto", "branco"
# }

# CORES = {
#     "preto", "preta", "branco", "branca", "azul", "rosa", "verde",
#     "vermelho", "vermelha", "cinza", "prata", "dourado", "gold",
#     "silver", "black", "white", "grafite", "lilas", "roxo", "bege"
# }

# MARCAS = [
#     "apple", "samsung", "xiaomi", "motorola", "lg", "philips",
#     "mondial", "electrolux", "consul", "brastemp", "lorenzetti",
#     "pampers", "huggies", "cerave", "cetaphil", "buba", "ype",
#     "pro tork", "loren shower", "acqua pure"
# ]


# def extrair_marca(texto):
#     for marca in sorted(MARCAS, key=len, reverse=True):
#         if marca in texto:
#             return marca
#     return None


# def extrair_armazenamento(texto):
#     # 128gb, 256 gb, 1tb etc.
#     m = re.search(r'(\d+)\s?(gb|tb)\b', texto)
#     if m:
#         return f"{m.group(1)}{m.group(2)}"
#     return None


# def extrair_ram(texto):
#     # tenta pegar algo tipo "8gb ram"
#     m = re.search(r'(\d+)\s?gb\s?ram\b', texto)
#     if m:
#         return f"{m.group(1)}gb"
#     return None


# def extrair_voltagem(texto):
#     m = re.search(r'(\d{2,3})\s?v\b', texto)
#     if m:
#         return f"{m.group(1)}v"
#     return None


# def extrair_potencia(texto):
#     m = re.search(r'(\d{3,5})\s?w\b', texto)
#     if m:
#         return f"{m.group(1)}w"
#     return None


# def extrair_quantidade(texto):
#     # ex.: 40 unidades, 48 unid, 500ml, 5l
#     m1 = re.search(r'(\d+)\s?(unidades|unidade|unid|unds)\b', texto)
#     if m1:
#         return f"{m1.group(1)}un"

#     m2 = re.search(r'(\d+)\s?(ml|l|kg|g)\b', texto)
#     if m2:
#         return f"{m2.group(1)}{m2.group(2)}"

#     return None


# def extrair_tamanho(texto):
#     # fralda / roupa / capacete etc.
#     padrao = r'\b(xg|xxg|xxgg|g|gg|m|p|rn|tam\s?[a-z0-9]+|tamanho\s?[a-z0-9]+)\b'
#     m = re.search(padrao, texto)
#     if m:
#         return m.group(1).replace("tam ", "").replace("tamanho ", "").strip()
#     return None


# def extrair_cor(texto):
#     tokens = set(texto.split())
#     for cor in CORES:
#         if cor in tokens:
#             return cor
#     return None


# def extrair_modelo(texto):
#     """
#     Estratégia simples:
#     remove marca, armazenamento, ram, cor, stopwords e tenta manter
#     os tokens mais informativos do título.
#     """
#     t = texto

#     marca = extrair_marca(t)
#     if marca:
#         t = t.replace(marca, " ")

#     # remove padrões conhecidos para não contaminar o "modelo"
#     t = re.sub(r'\b\d+\s?(gb|tb)\b', ' ', t)
#     t = re.sub(r'\b\d+\s?gb\s?ram\b', ' ', t)
#     t = re.sub(r'\b\d{2,3}\s?v\b', ' ', t)
#     t = re.sub(r'\b\d{3,5}\s?w\b', ' ', t)
#     t = re.sub(r'\b\d+\s?(unidades|unidade|unid|unds|ml|l|kg|g)\b', ' ', t)

#     tokens = []
#     for tok in t.split():
#         if tok not in STOPWORDS_TITULO and tok not in CORES:
#             tokens.append(tok)

#     # mantém até 5 tokens mais relevantes
#     tokens = tokens[:5]

#     if tokens:
#         return " ".join(tokens)
#     return None


# def extrair_atributos(texto):
#     texto = limpar_texto_attr(texto)

#     return {
#         "texto_limpo": texto,
#         "marca": extrair_marca(texto),
#         "armazenamento": extrair_armazenamento(texto),
#         "ram": extrair_ram(texto),
#         "voltagem": extrair_voltagem(texto),
#         "potencia": extrair_potencia(texto),
#         "quantidade": extrair_quantidade(texto),
#         "tamanho": extrair_tamanho(texto),
#         "cor": extrair_cor(texto),
#         "modelo": extrair_modelo(texto)
#     }


# def jaccard_tokens(txt1, txt2):
#     s1 = set(str(txt1).split())
#     s2 = set(str(txt2).split())

#     if not s1 and not s2:
#         return 0.0
#     return len(s1 & s2) / len(s1 | s2)


# def score_campo(v1, v2):
#     if v1 is None or v2 is None:
#         return None
#     return 1.0 if str(v1) == str(v2) else 0.0


# def similaridade_atributos(attr1, attr2):
#     """
#     Score ponderado.
#     Campos mais importantes recebem mais peso.
#     """
#     pesos = {
#         "marca": 0.20,
#         "modelo": 0.30,
#         "armazenamento": 0.15,
#         "ram": 0.10,
#         "voltagem": 0.08,
#         "potencia": 0.05,
#         "quantidade": 0.05,
#         "tamanho": 0.04,
#         "cor": 0.03
#     }

#     soma_pesos = 0.0
#     soma_score = 0.0

#     for campo, peso in pesos.items():
#         sc = score_campo(attr1.get(campo), attr2.get(campo))
#         if sc is not None:
#             soma_score += sc * peso
#             soma_pesos += peso

#     # complemento textual leve sobre o "modelo"
#     sim_modelo_textual = jaccard_tokens(
#         attr1.get("modelo", "") or "",
#         attr2.get("modelo", "") or ""
#     )

#     # entra como reforço
#     peso_modelo_textual = 0.20
#     soma_score += sim_modelo_textual * peso_modelo_textual
#     soma_pesos += peso_modelo_textual

#     if soma_pesos == 0:
#         return 0.0

#     return round(soma_score / soma_pesos, 3)

In [ ]:
# # =========================================================
# # 2) EXTRAÇÃO DE ATRIBUTOS
# # =========================================================

# amazon["atributos_extraidos"] = amazon[col_nome_amz].apply(extrair_atributos)
# mercadolivre["atributos_extraidos"] = mercadolivre[col_nome_ml].apply(extrair_atributos)

In [ ]:
# # =========================================================
# # 3) MATCHING POR ATRIBUTOS EXTRAÍDOS
# # =========================================================

# matches_attr = []

# for categoria_amz, categorias_ml in MAPA_CATEGORIAS.items():

#     amazon_cat = amazon[amazon[col_cat_amz] == categoria_amz]
#     ml_cat = mercadolivre[mercadolivre[col_cat_ml].isin(categorias_ml)]

#     if amazon_cat.empty or ml_cat.empty:
#         continue

#     for _, prod_amz in amazon_cat.iterrows():

#         melhor_score = -1
#         melhor_prod_ml = None

#         attr_amz = prod_amz["atributos_extraidos"]

#         for _, prod_ml in ml_cat.iterrows():
#             attr_ml = prod_ml["atributos_extraidos"]

#             score = similaridade_atributos(attr_amz, attr_ml)

#             if score > melhor_score:
#                 melhor_score = score
#                 melhor_prod_ml = prod_ml

#         if melhor_prod_ml is None:
#             continue

#         # Amazon
#         nota_amz = converter_nota(prod_amz[col_nota])
#         qtd_amz = converter_qtd(prod_amz[col_qtd])

#         # Mercado Livre
#         nota_ml = converter_nota(melhor_prod_ml[col_nota])
#         qtd_ml = converter_qtd(melhor_prod_ml[col_qtd])

#         nota_final = media_ponderada(
#             nota_amz, qtd_amz,
#             nota_ml, qtd_ml
#         )

#         asin_amz = str(prod_amz["ASIN"]).strip().upper()
#         asin_ml  = str(melhor_prod_ml.get("ASIN", "")).strip().upper()

#         preco_amz = lookup_preco_amz.get(asin_amz, None)
#         preco_ml  = lookup_preco_ml.get(asin_ml, None)

#         matches_attr.append({
#             "Categoria Amazon": categoria_amz,
#             "ASIN Amazon": prod_amz["ASIN"],
#             "Produto Amazon": prod_amz[col_nome_amz],
#             "Preço Prod Amazon": preco_amz,
#             "Nota Amazon": nota_amz,
#             "Avaliações Amazon": qtd_amz,
#             "Link Amazon": prod_amz["Link"],

#             "ASIN Mercado Livre": melhor_prod_ml.get("ASIN", None),
#             "Produto Mercado Livre": melhor_prod_ml[col_nome_ml],
#             "Preço Prod ML": preco_ml,
#             "Nota Mercado Livre": nota_ml,
#             "Avaliações ML": qtd_ml,
#             "Link Mercado Livre": melhor_prod_ml["Link"],

#             "Similaridade_Atributos": melhor_score,
#             "Nota Geral Ponderada": nota_final,

#             # opcional: guardar atributos para auditoria
#             "Atributos Amazon": attr_amz,
#             "Atributos ML": melhor_prod_ml["atributos_extraidos"]
#         })

In [ ]:
# # =========================================================
# # 4) RESULTADO FINAL DO MÉTODO DE ATRIBUTOS
# # =========================================================

# resultados_attr = pd.DataFrame(matches_attr)

# limiar_attr = 0.70
# resultados_attr_filtrados = resultados_attr[
#     resultados_attr["Similaridade_Atributos"] >= limiar_attr
# ].copy()

# print("🧾 Resultados filtrados - Atributos:")
# print(resultados_attr_filtrados.head())

🧾 Resultados filtrados - Atributos:
               Categoria Amazon ASIN Amazon  \
18          Alimentos e Bebidas  B00P9VQCXY   
214              Beleza Premium  B07XP8K162   
331                        Casa  B076BJMK4N   
418  Computadores e Informática  B0F1G2TL8P   
440                     Cozinha  B0DCKBMVH9   

                                        Produto Amazon Preço Prod Amazon  \
18      Azeite De Oliva Extra Virgem Andorinha - 500ml             35,90   
214  CeraVe Creme Hidratante Corporal, para Pele Se...             92,99   
331                            Ypê Detergente Clear 5L             28,79   
418  Mouse Sem Fio Recarregável Wireless Bluetooth ...             23,65   
440  Mixer Power Inox 3 em 1 600W Elgin Lunar 110V ...            115,90   

     Nota Amazon  Avaliações Amazon                              Link Amazon  \
18           4.9               9727  https://www.amazon.com.br/dp/B00P9VQCXY   
214          4.8               1182  https://www.amazon.com.br/d

In [ ]:
# # 5) Salva atributos
# atributos = f"data/comparacoes/atributos_{datetime.now().strftime('%Y-%m-%d_%H-%M')}.csv"
# resultados_attr_filtrados.to_csv(atributos, index=False, sep=';', encoding='utf-8-sig')

# print(f"\n🧩 Consolidado (atualizado). Total linhas: {len(resultados_attr_filtrados)}")
# print(f"✅ Atributos salvo em: {atributos}")


🧩 Consolidado (atualizado). Total linhas: 14
✅ Atributos salvo em: data/comparacoes/atributos_2026-03-10_17-00.csv
